## Data Availability Notice

The data used in this notebook was provided by **Dogs Trust** under a formal data sharing agreement as part of an MSc Data Science dissertation at Liverpool John Moores University, supervised by Dr Jon Bielby.

**The raw data files are not included in this repository.** To reproduce the analysis, the following data extracts from the Dogs Trust Salesforce CRM are required:

- `Dog Affiliation and Status_Sub Status History Pull.xlsx` (Affiliation History sheet)
- `Sub Status History v2.csv`

Requests for data access should be directed to the Dogs Trust Data Science & Analytics team.

All data handling was conducted in accordance with GDPR and Dogs Trust's data governance protocols.

In [ ]:
import simpy
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Load affiliation history
affiliation_history = pd.read_excel(
    '../data/[Dogs Trust data file - not included in repo]', 
    sheet_name='Affiliation History'
)

In [ ]:
#Simulation parameters from 02_simulation_inputs.ipynb
RANDOM_SEED = 42
SIM_DURATION = 365       # simulate 1 year (in days)
ARRIVAL_RATE = 628 / 30  # average dogs per month (628/month)
KENNEL_CAPACITY = 500    # starting assumption - to be tested in scenarios

# Intake duration - Gamma(shape=0.9319, scale=26.4569)
INTAKE_SHAPE = 0.9319
INTAKE_SCALE = 26.4569

# Movement duration - LogNormal(shape=1.0914, scale=11.2346)
MOVEMENT_SHAPE = 1.0914
MOVEMENT_SCALE = 11.2346

# Transtion probabilities from matrix
P_INTAKE_TO_MOVEMENT = 0.502
P_INTAKE_TO_ADOPTION = 0.420

P_MOVEMENT_TO_MOVEMENT = 0.387
P_MOVEMENT_TO_ADOPTION = 0.383

In [ ]:
# Development iteration v1 — baseline model (no transfer/return stages)
# Dog process function
def dog_journey(env, dog_id, results):
    arrival_time = env.now

    # INTAKE STAGE
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    # Decision: Intake to Movement or Intake to Adoption
    if np.random.random() < P_INTAKE_TO_ADOPTION:
        # Direct adoption - skip movement
        los = env.now - arrival_time
        results.append({
            'dog_id': dog_id,
            'arrival_time': arrival_time,
            'los': los,
            'outcome': 'Direct Adoption',
            'movement': 0
        })
        return

    # MOVEMENT STAGE (loop)
    movement_count = 0
    while True:
        movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
        yield env.timeout(movement_duration)
        movement_count += 1

        # Decision: another Movement or Adoption
        if np.random.random() < P_MOVEMENT_TO_ADOPTION / (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT):
            break

    # ADOPTION
    los = env.now - arrival_time
    results.append({
        'dog_id': dog_id,
        'arrival_time': arrival_time,
        'los': los,
        'outcome': 'Adopted',
        'movement': movement_count
    })

print("Dog process function defined")

In [ ]:
# Arrival generator
def arrival_generator(env, results):
    dog_id = 0
    while True:
        # Inter-arrival time - Exponential based on daily arrival rate
        inter_arrival = np.random.exponential(1 / ARRIVAL_RATE)
        yield env.timeout(inter_arrival)
        dog_id += 1
        env.process(dog_journey(env, dog_id, results))

# Run simulation
np.random.seed(RANDOM_SEED)       
results = []

env = simpy.Environment()
env.process(arrival_generator(env, results))
env.run(until=SIM_DURATION)

# Convert results to dataframe
results_df = pd.DataFrame(results)

print(f"Simulation complete")
print(f"Total dogs processed: {len(results_df)}")
print(f"\nSummary statistics:")
print(results_df[['los', 'movement']].describe().round(2))

In [ ]:
# Monte Carlo replications
N_REPLICATIONS = 100
all_results = []

for rep in range(N_REPLICATIONS):
    np.random.seed(rep)  # different seed each run
    rep_results = []

    env = simpy.Environment()
    env.process(arrival_generator(env, rep_results))
    env.run(until=SIM_DURATION)

    rep_df = pd.DataFrame(rep_results)
    rep_df['replication'] = rep
    all_results.append(rep_df)

# Combine all replications
mc_df = pd.concat(all_results, ignore_index=True)

print(f"Total replications: {N_REPLICATIONS}")
print(f"Total dogs across all replications: {len(mc_df)}")
print(f"\nMean LOS per replication (sample):")
print(mc_df.groupby('replication')['los'].mean().describe().round(2))

In [ ]:
# Summary statistics across replications
rep_summary = mc_df.groupby('replication').agg(
    mean_los=('los', 'mean'),
    median_los=('los', 'median'),
    mean_movements=('movement', 'mean'),
    total_dogs=('dog_id', 'count')
).reset_index()

# 95% confidence intervals
ci_95 = rep_summary[['mean_los', 'median_los', 'mean_movements']].agg(
    ['mean', 'std']
).T

ci_95['95% CI Lower'] = ci_95['mean'] - 1.96 * ci_95['std']
ci_95['95% CI Upper'] = ci_95['mean'] + 1.96 * ci_95['std']
ci_95 = ci_95.round(2)

print("Monte Carlo Results - 95% Confidence Intervals:")
print(ci_95[['mean', '95% CI Lower', '95% CI Upper']])

In [ ]:
# Monte Carlo LOS distribution
fig_mc = px.histogram(
    mc_df,
    x='los',
    nbins=50,
    title='Monte Carlo LOS Distribution - 100 Replications (Simple Journey)',
    labels={'los': 'Length of Stay (days)', 'count': 'Frequency'},
    color_discrete_sequence=['steelblue'],
    opacity=0.7
)

# Add mean and CI lines
fig_mc.add_vline(x=40.51, line_dash='solid', line_color='red',
    annotation_text='Mean: 40.51 days', 
    annotation_position='top right',
    annotation_y=0.95)

fig_mc.add_vline(x=39.59, line_dash='dash', line_color='orange',
    annotation_text='Lower CI: 39.59 days',
    annotation_position='top left',
    annotation_y=0.80)

fig_mc.add_vline(x=41.44, line_dash='dash', line_color='orange',
    annotation_text='Upper CI: 41.44 days',
    annotation_position='top right',
    annotation_y=0.80)

fig_mc.update_layout(height=500, width=900)
fig_mc.show()

In [ ]:
fig_mc.write_image('../outputs/monte_carlo_los_distribution.png', width=1100, height=600, scale=2)
fig_mc.write_html('../outputs/monte_carlo_los_distribution.html')

In [ ]:
# Band simulation outputs into LOS bands (Dogs Trust operational bands)
bins = [0, 60, 182, float('inf')]
labels = ['0-60 days (Normal)', '60-182 days (Medium)', '183+ days (Long-term)']

mc_df['LOS Band'] = pd.cut(mc_df['los'], bins=bins, labels=labels)

# Simulated band proportions
sim_band_props = mc_df['LOS Band'].value_counts(normalize=True).sort_index() * 100

# Real data band proportions (Dogs Trust bands, from 01_data_exploration)
real_band_props = pd.Series({
    '0-60 days (Normal)': 71.6,
    '60-182 days (Medium)': 16.7,
    '183+ days (Long-term)': 7.3
})

comparison = pd.DataFrame({
    'Real Data (%)': real_band_props,
    'Simulation (%)': sim_band_props.round(1)
})

print("Validation - LOS Band Comparison:")
print(comparison)

In [ ]:
fig_val = px.bar(
    comparison.reset_index(),
    x='index',
    y=['Real Data (%)', 'Simulation (%)'],
    barmode='group',
    title='Validation - Real vs Simulated LOS Band Proportions',
    labels={'index': 'LOS Band', 'value': 'Proportion (%)', 'variable': 'Source'},
    color_discrete_sequence=['steelblue', 'salmon']
)

fig_val.update_layout(height=500, width=900)
fig_val.write_image('../outputs/validation_los_band_comparison.png', width=1100, height=600, scale=2)
fig_val.write_html('../outputs/validation_los_band_comparison.html') #Interactive session
fig_val.show()

In [ ]:
# Transfer duration - from stage duration analysis
# Transfer had mean 18 days, median 1 day - fit Exponential
TRANSFER_RATE = 1 / 18 # rate parameter for Exponential

# Updated transition probabilities including Transfer
P_INTAKE_TO_TRANSFER = 0.077

P_MOVEMENT_TO_TRANSFER = 0.167

P_TRANSFER_TO_MOVEMENT = 0.357
P_TRANSFER_TO_ADOPTION = 0.327
P_TRANSFER_TO_TRANSFER = 0.304

In [ ]:
# Development iteration v2 — adds transfer stage
def transfer_stage(env):
    transfer_duration = np.random.exponential(1 / TRANSFER_RATE)
    yield env.timeout(transfer_duration)

def dog_journey(env, dog_id, results):
    arrival_time = env.now

    # INTAKE STAGE
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    # Decision after Intake
    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'arrival_time': arrival_time, 
                        'los': los, 'outcome': 'Direct Adoption', 'movement': 0})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        # Go to Transfer
        yield from transfer_stage(env)

    # MOVEMENT STAGE (loop)
    movement_count = 0
    while True:
        movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
        yield env.timeout(movement_duration)
        movement_count += 1

        rand = np.random.random()
        if rand < P_MOVEMENT_TO_ADOPTION / (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER):
            break
        elif rand < (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_TRANSFER) / (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER):
            yield from transfer_stage(env)


    # ADOPTION
    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'arrival_time': arrival_time, 
                    'los': los, 'outcome': 'Adopted', 'movement': movement_count})
                   


In [ ]:
# Run simulation with Transfer journey
N_REPLICATIONS = 100
all_results = []

for rep in range(N_REPLICATIONS):
    np.random.seed(rep)
    rep_results = []

    env = simpy.Environment()
    env.process(arrival_generator(env, rep_results))
    env.run(until=SIM_DURATION)

    rep_df = pd.DataFrame(rep_results)
    rep_df['replication'] = rep
    all_results.append(rep_df)

mc_df = pd.concat(all_results, ignore_index=True)

print(f"Total dogs across all replications: {len(mc_df)}")
print(f"\nMean LOS per replication:")
print(mc_df.groupby('replication')['los'].mean().describe().round(2))

In [ ]:
# Band simulation outputs (Dogs Trust operational bands)
mc_df['LOS Band'] = pd.cut(mc_df['los'], bins=bins, labels=labels)

sim_band_props = mc_df['LOS Band'].value_counts(normalize=True).sort_index() * 100

comparison_v2 = pd.DataFrame({
    'Real Data (%)': real_band_props,
    'With Transfer (%)': sim_band_props.round(1)
})

print("Validation - LOS Band Comparison (With Transfer):")
print(comparison_v2)

In [ ]:
# Return journey parameters
P_ADOPTION_TO_RETURN = 0.150   # using conservative estimate (as most Adoption records have no end date)

P_RETURN_TO_ADOPTION = 0.561
P_RETURN_TO_MOVEMENT = 0.408

# Return duration - Exponential (mean 41.9 days)
RETURN_RATE = 1 / 41.9


In [ ]:
def transfer_stage(env):
    transfer_duration = np.random.exponential(1 / TRANSFER_RATE)
    yield env.timeout(transfer_duration)

def return_stage(env):
    return_duration = np.random.exponential(1 / RETURN_RATE)    
    yield env.timeout(return_duration)

def dog_journey(env, dog_id, results):
    arrival_time = env.now

    # INTAKE STAGE
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    # Decision after Intake
    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        # Check for Return after adoption
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            # After return - Movement then Adoption
            movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
            yield env.timeout(movement_duration)
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'arrival_time': arrival_time, 
                        'los': los, 'outcome': 'Adopted', 'movement': 0})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    # MOVEMEMT STAGE (loop)
    movement_count = 0
    while True:
        movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
        yield env.timeout(movement_duration)
        movement_count += 1

        rand = np.random.random()
        if rand < P_MOVEMENT_TO_ADOPTION / (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER):
            # Check for Return after adoption
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
                yield env.timeout(movement_duration)
                movement_count += 1
            break
        elif rand < (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_TRANSFER) / (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER):
            yield from transfer_stage(env)

    # ADOPTION
    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'arrival_time': arrival_time, 
                    'los': los, 'outcome': 'Adopted', 'movement': movement_count})    

In [ ]:
# Run simulation with Return journey
N_REPLICATIONS = 100
all_results = []

for rep in range(N_REPLICATIONS):
    np.random.seed(rep)
    rep_results = []

    env = simpy.Environment()
    env.process(arrival_generator(env, rep_results))
    env.run(until=SIM_DURATION)

    rep_df = pd.DataFrame(rep_results)
    rep_df['replication'] = rep
    all_results.append(rep_df)

mc_df = pd.concat(all_results, ignore_index=True)

print(f"Total dogs across all replications: {len(mc_df)}")
print(f"\nMean LOS per replication:")
print(mc_df.groupby('replication')['los'].mean().describe().round(2))

In [ ]:
mc_df['LOS Band'] = pd.cut(mc_df['los'], bins=bins, labels=labels)
sim_band_props_v3 = mc_df['LOS Band'].value_counts(normalize=True).sort_index() * 100

comparison_v3 = pd.DataFrame({
    'Real Data (%)': real_band_props,
    'With Transfer + Return (%)': sim_band_props_v3.round(1)
})

print("Validation - LOS Band Comparison (With Transfer + Return):")
print(comparison_v3)

In [ ]:
# Combine both comparisons into one dataframe
comparison_combined = pd.DataFrame({
    'Real Data (%)': real_band_props,
    'With Transfer (%)': comparison_v2['With Transfer (%)'],
    'With Transfer + Return (%)': comparison_v3['With Transfer + Return (%)']
})

fig_val_combined = px.bar(
    comparison_combined.reset_index(),
    x='index',
    y=['Real Data (%)', 'With Transfer (%)', 'With Transfer + Return (%)'],
    barmode='group',
    title='Validation — Real vs Simulated LOS Bands (Progressive Model Build, Dogs Trust Bands)',
    labels={'index': 'LOS Band', 'value': 'Proportion (%)', 'variable': 'Model Version'},
    color_discrete_sequence=['steelblue', 'green', 'purple']
)

fig_val_combined.update_layout(height=500, width=1000, legend=dict(x=0.6, y=1.15))
fig_val_combined.write_image('../outputs/validation_progressive_model_build.png', width=1200, height=600, scale=2)
fig_val_combined.write_html('../outputs/validation_progressive_model_build.html')
fig_val_combined.show()

In [ ]:
def dog_journey_capacity(env, dog_id, kennels, results):
    arrival_time = env.now

    with kennels.request() as request:
        yield request
        wait_time = env.now - arrival_time

        # INTAKE STAGE
        intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
        yield env.timeout(intake_duration)

        # Decision after Intake
        rand = np.random.random()
        if rand < P_INTAKE_TO_ADOPTION:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
                yield env.timeout(movement_duration)
            los = env.now - arrival_time
            results.append({'dog_id': dog_id, 'arrival_time': arrival_time,
                        'los': los, 'wait_time': wait_time, 'outcome': 'Direct Adoption'})
            return
        elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
            yield from transfer_stage(env)

        # MOVEMENT STAGE
        movement_count = 0
        while True:
            movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
            yield env.timeout(movement_duration)
            movement_count += 1

            rand = np.random.random()
            if rand < P_MOVEMENT_TO_ADOPTION / (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER):
                if np.random.random() < P_ADOPTION_TO_RETURN:
                    yield from return_stage(env)
                    movement_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
                    yield env.timeout(movement_duration)
                    movement_count += 1
                break
            elif rand < (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_TRANSFER) / (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER):
                yield from transfer_stage(env)

        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'arrival_time': arrival_time,
                        'los': los, 'wait_time': wait_time, 'outcome': 'Adopted'})


def arrival_generator_capacity(env, kennels, results):
    dog_id = 0
    while True:
        inter_arrival = np.random.exponential(1 / ARRIVAL_RATE)
        yield env.timeout(inter_arrival)
        dog_id += 1
        env.process(dog_journey_capacity(env, dog_id, kennels, results))

# RQ2 - Kennel Capacity Sweep
# Test different kennel capacities and measure LOS and waiting times

capacities = [200, 400, 600, 800, 1000]
N_REPLICATIONS = 100
capacity_results = []

for capacity in capacities:
    rep_los = []
    rep_wait = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        kennels = simpy.Resource(env, capacity=capacity)
        env.process(arrival_generator_capacity(env, kennels, rep_results))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        if len(rep_df) > 0:
            rep_los.append(rep_df['los'].mean())
            rep_wait.append(rep_df['wait_time'].mean())


    capacity_results.append({
        'Capacity': capacity,
        'Mean LOS': np.mean(rep_los).round(2),
        'LOS 95% CI Lower': (np.mean(rep_los) - 1.96 * np.std(rep_los)).round(2),
        'LOS 95% CI Upper': (np.mean(rep_los) + 1.96 * np.std(rep_los)).round(2),
        'Mean Wait Time': np.mean(rep_wait).round(2),
        'Wait 95% CI Lower': (np.mean(rep_wait) - 1.96 * np.std(rep_wait)).round(2),
        'Wait 95% CI Upper': (np.mean(rep_wait) + 1.96 * np.std(rep_wait)).round(2),
    })
    print(f"Capacity {capacity} done - Mean LOS: {np.mean(rep_los):.2f}, Mean Wait: {np.mean(rep_wait):.2f}")

capacity_df = pd.DataFrame(capacity_results)
print("\nCapacity Sweep Results:")
print(capacity_df.to_string(index=False))

In [ ]:
fig_capacity = go.Figure()

# LOS line
fig_capacity.add_trace(go.Scatter(
    x=capacity_df['Capacity'],
    y=capacity_df['Mean LOS'],
    mode='lines+markers',
    name='Mean LOS',
    line=dict(color='steelblue', width=2),
    error_y=dict(
        type='data',
        symmetric=False,
        array=capacity_df['LOS 95% CI Upper'] - capacity_df['Mean LOS'],
        arrayminus=capacity_df['Mean LOS'] - capacity_df['LOS 95% CI Lower'],
        visible=True
    )
))

# Wait Time line
fig_capacity.add_trace(go.Scatter(
    x=capacity_df['Capacity'],
    y=capacity_df['Mean Wait Time'],
    mode='lines+markers',
    name='Mean Wait Time',
    line=dict(color='salmon', width=2),
    error_y=dict(
        type='data',
        symmetric=False,
        array=capacity_df['Wait 95% CI Upper'] - capacity_df['Mean Wait Time'],
        arrayminus=capacity_df['Mean Wait Time'] - capacity_df['Wait 95% CI Lower'],
        visible=True
    )
))

fig_capacity.update_layout(
    title= 'RQ2 - Effect of Kennel Capacity on LOS and Waiting Time (100 replications)',
    xaxis_title='Kennel Capacity (spaces)',
    yaxis_title='Days',
    height=500,
    width=900,
    legend=dict(x=0.7, y=0.95)
)

fig_capacity.write_image('../outputs/rq2_capacity_sweep.png', width=1100, height=600, scale=2)
fig_capacity.write_html('../outputs/rq2_capacity_sweep.html')

fig_capacity.show()

In [ ]:
def dog_journey_foster(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer):
    arrival_time = env.now

    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los, 'outcome': 'Adopted'})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1

        rand = np.random.random()
        if rand < p_move_adopt:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < p_move_adopt + p_move_transfer:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los, 'outcome': 'Adopted'})


def arrival_gen_foster(env, results, p_move_adopt, p_move_move, p_move_transfer):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_foster(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer))

# RQ3 Scenario 1 - expand Home from Home (Foster Network)
# Assumption: foster placements increase P(Movement to Adoption) by reducing the Movement loop probability
# Baseline foster proportion: ~50% of Movement records
# We test by increasing this to 60%, 70%, 80%

# In the model, expanding foster is represented as:
# Higher P(Movement to Adoption) - Dogs in foster get adopted faster
# Lower P(Movement to Movement) - Lesser kennel shuffling

foster_scenarios = {
    'Baseline (50% foster)': {'p_move_adopt': 0.383, 'p_move_move': 0.387},
    'Expanded (60% foster)': {'p_move_adopt': 0.420, 'p_move_move': 0.350},
    'High Expansion (70% foster)': {'p_move_adopt': 0.460, 'p_move_move': 0.310},
    'Full Expansion (80% foster)': {'p_move_adopt': 0.500, 'p_move_move': 0.270},
}

N_REPLICATIONS = 100
foster_results = []

for scenario_name, params in foster_scenarios.items():
    p_move_adopt = params['p_move_adopt']
    p_move_move = params['p_move_move']
    p_move_transfer = 1 - p_move_adopt - p_move_move

    rep_los = []
    rep_adopted = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_foster(env, rep_results, p_move_adopt, p_move_move, p_move_transfer))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())
        rep_adopted.append(len(rep_df))

    foster_results.append({
        'Scenario': scenario_name,
        'P(Move-Adopt)': round(p_move_adopt, 3),
        'P(Move-Move)': round(p_move_move, 3),
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Dogs Rehomed': round(np.mean(rep_adopted), 0),
    })
    print(f"{scenario_name} done - Mean LOS: {np.mean(rep_los):.2f}, Dogs Rehomed: {np.mean(rep_adopted):.0f}")

foster_df = pd.DataFrame(foster_results)
print("\nRQ3 Scenario 1 - Foster Network Expansion Results:")
print(foster_df.to_string(index=False))

In [ ]:
fig_foster = go.Figure()

fig_foster.add_trace(go.Bar(
    x=foster_df['Scenario'],
    y=foster_df['Mean LOS'],
    name='Mean LOS (days)',
    marker_color='steelblue',
    error_y=dict(
        type='data',
        symmetric=False,
        array=foster_df['LOS 95% CI Upper'] - foster_df['Mean LOS'],
        arrayminus=foster_df['Mean LOS'] - foster_df['LOS 95% CI Lower'],
        visible=True
    )
))

fig_foster.add_trace(go.Scatter(
    x=foster_df['Scenario'],
    y=foster_df['Mean Dogs Rehomed'],
    name='Dogs Rehomed per Year',
    mode='lines+markers',
    line=dict(color='red', width=2),
    yaxis='y2'
))

fig_foster.update_layout(
    title='RQ3 Scenario 1 - Foster Network Expansion: LOS and Rehoming Throughput',
    xaxis_title='Scenario',
    yaxis_title='Mean LOS (days)',
    yaxis2=dict(
        title='Dogs Rehomed per Year',
        overlaying='y',
        side='right'
    ), 
    height=500,
    width=950,
    legend=dict(x=0.65, y=1.15),
    barmode='group'
)

fig_foster.write_image('../outputs/rq3_scenario1_foster_expansion.png', width=1100, height=600, scale=2)
fig_foster.write_html('../outputs/rq3_scenario1_foster_expansion.html')

fig_foster.show()

In [ ]:
def dog_journey_priority(env, dog_id, results, threshold, boost):
    arrival_time = env.now

    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los, 'outcome': 'Adopted'})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1

        # Apply boost if dog has exceeded movement threshold
        if movement_count >= threshold:
            p_adopt = min(P_MOVEMENT_TO_ADOPTION + boost, 0.90)
            p_move = P_MOVEMENT_TO_MOVEMENT - boost / 2
            p_transfer = P_MOVEMENT_TO_TRANSFER - boost / 2
        else:
            p_adopt = P_MOVEMENT_TO_ADOPTION
            p_move = P_MOVEMENT_TO_MOVEMENT
            p_transfer = P_MOVEMENT_TO_TRANSFER

        # Normalise
        total = p_adopt + p_move + p_transfer
        rand = np.random.random()
        if rand < p_adopt / total:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < (p_adopt + p_transfer) / total:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los, 'outcome': 'Adopted'})


def arrival_gen_priority(env, results, threshold, boost):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_priority(env, dog_id, results, threshold, boost))

# RQ3 Scenario 2 - Longest-Stay Prioritisation
# Dogs that have been moved more times (proxy for longer stay) gets a boosted P (Movement - Adoption)
# Threshold: after N movements, adoption probability increases

priority_scenarios = {
    'Baseline (no priority)': {'threshold': 999, 'boost': 0.0},
    'Priority after 2 moves': {'threshold': 2, 'boost': 0.15},
    'Priority after 3 moves': {'threshold': 3, 'boost': 0.15},
    'Priority after 2 moves (high boost)': {'threshold': 2, 'boost': 0.25},
}

N_REPLICATIONS = 100
priority_results = []

for scenario_name, params in priority_scenarios.items():
    threshold = params['threshold']
    boost = params['boost']

    rep_los = []
    rep_adopted = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_priority(env, rep_results, threshold, boost))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())
        rep_adopted.append(len(rep_df))

    priority_results.append({
        'Scenario': scenario_name,
        'Threshold (moves)': threshold if threshold != 999 else 'None',
        'Adoption Boost': boost,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Dogs Rehomed': round(np.mean(rep_adopted), 0),
    })
    print(f"{scenario_name} done - Mean LOS: {np.mean(rep_los):.2f}, Dogs Rehomed: {np.mean(rep_adopted):.0f}")

priority_df = pd.DataFrame(priority_results)
print("\nRQ3 Scenario 2 - Longest-Stay Prioritisation Results:")
print(priority_df.to_string(index=False))

In [ ]:
fig_priority = go.Figure()

fig_priority.add_trace(go.Bar(
    x=priority_df['Scenario'],
    y=priority_df['Mean LOS'],
    name='Mean LOS (days)',
    marker_color='steelblue',
    error_y=dict(
        type='data',
        symmetric=False,
        array=priority_df['LOS 95% CI Upper'] - priority_df['Mean LOS'],
        arrayminus=priority_df['Mean LOS'] - priority_df['LOS 95% CI Lower'],
        visible=True
    )
))

fig_priority.add_trace(go.Scatter(
    x=priority_df['Scenario'],
    y=priority_df['Mean Dogs Rehomed'],
    name='Dogs Rehomed per Year',
    mode='lines+markers',
    line=dict(color='red', width=2),
    yaxis='y2'
))

fig_priority.update_layout(
    title='RQ3 Scenario 2 - Longest-Stay Prioritisation: LOS and Rehoming Throughput',
    xaxis_title='Scenario',
    yaxis_title='Mean LOS (days)',
    yaxis2=dict(
        title='Dogs Rehomed per Year',
        overlaying='y',
        side='right'
    ),
    height=500,
    width=950,
    legend=dict(x=0.65, y=1.15),
    barmode='group' 
)
fig_priority.write_image('../outputs/rq3_scenario2_priority.png', width=1100, height=600, scale=2)
fig_priority.write_html('../outputs/rq3_scenario2_priority.html')

fig_priority.show()

In [ ]:
def dog_journey_combined(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost):
    arrival_time = env.now

    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < P_ADOPTION_TO_RETURN:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los, 'outcome': 'Adopted'})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1

        if movement_count >= threshold:
            p_adopt = min(p_move_adopt + boost, 0.90)
            p_move = p_move_move - boost / 2
            p_trans = p_move_transfer - boost / 2
        else:
            p_adopt = p_move_adopt
            p_move = p_move_move
            p_trans = p_move_transfer

        total = p_adopt + p_move + p_trans
        rand = np.random.random()
        if rand < p_adopt / total:
            if np.random.random() < P_ADOPTION_TO_RETURN:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < (p_adopt + p_trans) / total:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los, 'outcome': 'Adopted'})


def arrival_gen_combined(env, results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_combined(env, dog_id, results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost))

# RQ3 Scenario 3 - Combined Intervention
# Full foster expansion (80%) + longest-stay prioritisation (after 2 moves, high boost)

combined_scenarios = {
    'Baseline': {
        'p_move_adopt': 0.383, 'p_move_move': 0.387,
        'threshold': 999, 'boost': 0.0
    },
    'Foster Only (80%)': {
        'p_move_adopt': 0.500, 'p_move_move': 0.270,
        'threshold': 999, 'boost': 0.0
    },
    'Priority Only (high boost)': {
        'p_move_adopt': 0.383, 'p_move_move': 0.387,
        'threshold': 2, 'boost': 0.25
    },
    'Combined (Foster + Priority)': {
        'p_move_adopt': 0.500, 'p_move_move': 0.270,
        'threshold': 2, 'boost': 0.25
    },
}

N_REPLICATIONS = 100
combined_results = []

for scenario_name, params in combined_scenarios.items():
    p_move_adopt = params['p_move_adopt']
    p_move_move = params['p_move_move']
    p_move_transfer = 1 - p_move_adopt - p_move_move
    threshold = params['threshold']
    boost = params['boost']

    rep_los = []
    rep_adopted = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_combined(env, rep_results, p_move_adopt, p_move_move, p_move_transfer, threshold, boost))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())
        rep_adopted.append(len(rep_df))

    combined_results.append({
        'Scenario': scenario_name,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
        'Mean Dogs Rehomed': round(np.mean(rep_adopted), 0),
        'Extra Dogs Rehomed': round(np.mean(rep_adopted) - 6204, 0),
        'LOS Reduction': round(52.24 - np.mean(rep_los), 2)
    })
    print(f"{scenario_name} done - Mean LOS: {np.mean(rep_los):.2f}, Dogs Rehomed: {np.mean(rep_adopted):.0f}")

combined_df = pd.DataFrame(combined_results)
print("\nRQ3 Scenario 3 - Combined Intervention Results:")
print(combined_df.to_string(index=False))

In [ ]:
fig_combined = go.Figure()

fig_combined.add_trace(go.Bar(
    x=combined_df['Scenario'],
    y=combined_df['Mean LOS'],
    name='Mean LOS (days)',
    marker_color='steelblue',
    error_y=dict(
        type='data',
        symmetric=False,
        array=combined_df['LOS 95% CI Upper'] - combined_df['Mean LOS'],
        arrayminus=combined_df['Mean LOS'] - combined_df['LOS 95% CI Lower'],
        visible=True
    )
))

fig_combined.add_trace(go.Scatter(
    x=combined_df['Scenario'],
    y=combined_df['Mean Dogs Rehomed'],
    name='Dogs Rehomed per Year',
    mode='lines+markers',
    line=dict(color='red', width=2),
    yaxis='y2'
))

fig_combined.update_layout(
    title='RQ3 Scenario 3 - Combined Intervention vs Individual Scenarios',
    xaxis_title='Scenario',
    yaxis_title='Mean LOS (days)',
    yaxis2=dict(
        title='Dog Rehomed per Year',
        overlaying='y',
        side='right'
    ),
    height=500,
    width=950,
    legend=dict(x=0.65, y=1.15),
)
fig_combined.write_image('../outputs/rq3_scenario3_combined.png', width=1100, height=600, scale=2)
fig_combined.write_html('../outputs/rq3_scenario3_combined.html')

fig_combined.show()

In [ ]:
# Recreate los_df for QQ plot validation
intake_dates = affiliation_history[affiliation_history['Process'] == 'Intake']\
    .sort_values('Start Date')\
    .groupby('Dog ID')['Start Date'].first().reset_index()
intake_dates.columns = ['Dog ID', 'Intake Date']

last_records = affiliation_history.sort_values('Start Date')\
    .groupby('Dog ID').last().reset_index()
last_records['Exit Date'] = last_records['End Date'].fillna(last_records['Start Date'])

los_df = intake_dates.merge(last_records[['Dog ID', 'Exit Date']], on='Dog ID')
los_df['Length of Stay'] = (
    pd.to_datetime(los_df['Exit Date']) - pd.to_datetime(los_df['Intake Date'])
).dt.days


In [ ]:
# QQ Plot - Real vs Simulated LOS
real_los = los_df['Length of Stay'].dropna().values
real_los = real_los[real_los >= 0]

sim_los = mc_df['los'].sample(n=len(real_los), random_state=42).values

real_sorted = np.sort(real_los)
sim_sorted = np.sort(sim_los)

min_len = min(len(real_sorted), len(sim_sorted))
real_sorted = real_sorted[:min_len]
sim_sorted = sim_sorted[:min_len]

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(real_sorted, sim_sorted, alpha=0.3, s=5, color='steelblue', label='Quantile pairs')
ax.plot([0, max(real_sorted)], [0, max(real_sorted)], 'r--', lw=2, label='Perfect fit (y=x)')

# Shade good fit region (0-60 days = Normal band)
ax.axvspan(0, 60, alpha=0.08, color='green', label='Good fit region (0–60 days, Normal)')

# Annotations
ax.annotate('Model closely follows\nreal data in this range\n(0–60 days, Normal)',
            xy=(30, 50), xytext=(120, 30),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=9, color='green')

ax.annotate('Simulation underestimates\nmedium-stay dogs (60–182 days)',
            xy=(120, 100), xytext=(300, 60),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red')

ax.annotate('Model rarely produces\nthe very longest stays\n(183+ days, Long-term)',
            xy=(500, 250), xytext=(400, 320),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=9, color='red')

ax.set_xlabel('Real LOS (days)', fontsize=12)
ax.set_ylabel('Simulated LOS (days)', fontsize=12)
ax.set_title('QQ Plot — Real vs Simulated LOS Distribution\n(Transfer + Return Journey Model, 100 Monte Carlo Replications)', fontsize=12)
ax.legend(fontsize=9)

# Subtitle text box
ax.text(0.01, 0.99,
    'Points on the red line = perfect match\nPoints below = simulation underestimates LOS\nPoints above = simulation overestimates LOS',
    transform=ax.transAxes, fontsize=8, verticalalignment='top',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('../outputs/qq_plot_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# One-sample t-tests vs conservative baseline mean (53.65 days)
baseline_means = mc_df.groupby('replication')['los'].mean().values
print(f"Baseline mean LOS across replications: {baseline_means.mean():.2f}")

# For each scenario, compare against baseline using the per-replication std and mean
# Use a one-sample t-test against baseline mean

scenarios = {
    'Foster Only (80%)': (50.04, 0.60),   # mean, std from results
    'Priority Only (high boost)': (50.33, 0.58),
    'Combined (Foster + Priority)': (47.89, 0.55)
}

print("\nOne-sample t-tests vs Baseline mean (53.65 days):")
baseline_mean = 53.65

for name, (mean, std) in scenarios.items():
    # t-statistic = (scenario_mean - baseline_mean) / (std / sqrt(n))
    n = 100
    t_stat = (mean - baseline_mean) / (std / np.sqrt(n))
    p_value = stats.t.sf(abs(t_stat), df=n-1) * 2 # two-tailed
    print(f"{name}: t={t_stat:.3f}, p={p_value:.6f} {'Significant (p<0.05)' if p_value < 0.05 else 'Not significant'}")

In [ ]:
significance_df = pd.DataFrame({
    'Scenario': ['Foster Only (80%)', 'Priority Only (high boost)', 'Combined (Foster + Priority)'],
    'Mean LOS': [50.04, 50.33, 47.89],
    'LOS Reduction vs Baseline': [3.61, 3.32, 5.76],
    'Extra Dogs Rehomed': [169, 188, 286],
    't-statistic': [-60.167, -57.241, -104.727],
    'p-value': ['<0.001', '<0.001', '<0.001'],
    'Significant': ['Yes', 'Yes', 'Yes']
})

print("RQ3 Significance Test Summary:")
print(significance_df.to_string(index=False))

In [ ]:
def dog_journey_sensitivity(env, dog_id, results, p_return):
    arrival_time = env.now

    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    rand = np.random.random()
    if rand < P_INTAKE_TO_ADOPTION:
        if np.random.random() < p_return:
            yield from return_stage(env)
            yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        los = env.now - arrival_time
        results.append({'dog_id': dog_id, 'los': los})
        return
    elif rand < P_INTAKE_TO_ADOPTION + P_INTAKE_TO_TRANSFER:
        yield from transfer_stage(env)

    movement_count = 0
    while True:
        yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
        movement_count += 1

        rand = np.random.random()
        total = P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_MOVEMENT + P_MOVEMENT_TO_TRANSFER
        if rand < P_MOVEMENT_TO_ADOPTION / total:
            if np.random.random() < p_return:
                yield from return_stage(env)
                yield env.timeout(stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE))
            break
        elif rand < (P_MOVEMENT_TO_ADOPTION + P_MOVEMENT_TO_TRANSFER) / total:
            yield from transfer_stage(env)

    los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': los})


def arrival_gen_sensitivity(env, results, p_return):
    dog_id = 0
    while True:
        yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
        dog_id += 1
        env.process(dog_journey_sensitivity(env, dog_id, results, p_return))

# Sensitivity Analysis - P_ADOPTION_TO_RETURN
# Test range: 0.05 to 0.30 in steps of 0.05

return_probs = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
N_REPLICATIONS = 100
sensitivity_results = []

for p_return in return_probs:
    rep_los = []

    for rep in range(N_REPLICATIONS):
        np.random.seed(rep)
        rep_results = []

        env = simpy.Environment()
        env.process(arrival_gen_sensitivity(env, rep_results, p_return))
        env.run(until=SIM_DURATION)

        rep_df = pd.DataFrame(rep_results)
        rep_los.append(rep_df['los'].mean())

    sensitivity_results.append({
        'P(Adoption-Return)': p_return,
        'Mean LOS': round(np.mean(rep_los), 2),
        'LOS 95% CI Lower': round(np.mean(rep_los) - 1.96 * np.std(rep_los), 2),
        'LOS 95% CI Upper': round(np.mean(rep_los) + 1.96 * np.std(rep_los), 2),
    })
    print(f"P(Return)={p_return} done - Mean LOS: {np.mean(rep_los):.2f}")

sensitivity_df = pd.DataFrame(sensitivity_results)
print("\nSensitivity Analysis Results:")
print(sensitivity_df.to_string(index=False))

In [ ]:
# DIAGNOSTIC: Multiple Return Cycles
# Testing whether allowing multiple return cycles explains the underrepresentation
# of Long-term dogs (183+ days) in the simulated LOS distribution

def dog_journey_multi_return(env, dog_id, results):
    arrival_time = env.now
    movement_count = 0

    # INTAKE STAGE
    intake_duration = stats.gamma.rvs(a=INTAKE_SHAPE, scale=INTAKE_SCALE)
    yield env.timeout(intake_duration)

    # Normalise intake probabilities to ensure they sum to 1
    intake_probs = np.array([P_INTAKE_TO_MOVEMENT, P_INTAKE_TO_ADOPTION, P_INTAKE_TO_TRANSFER])
    intake_probs = intake_probs / intake_probs.sum()
    current_stage = np.random.choice(['Movement', 'Adoption', 'Transfer'], p=intake_probs)

    # Allow multiple return cycles (up to 5)
    max_returns = 5
    return_count = 0

    while True:
        if current_stage == 'Movement':
            movement_count += 1
            move_duration = stats.lognorm.rvs(s=MOVEMENT_SHAPE, scale=MOVEMENT_SCALE)
            yield env.timeout(move_duration)

            # Normalise movement probabilities
            move_probs = np.array([P_MOVEMENT_TO_MOVEMENT, P_MOVEMENT_TO_ADOPTION, P_MOVEMENT_TO_TRANSFER])
            move_probs = move_probs / move_probs.sum()
            current_stage = np.random.choice(['Movement', 'Adoption', 'Transfer'], p=move_probs)

        elif current_stage == 'Transfer':
            transfer_duration = np.random.exponential(1 / TRANSFER_RATE)
            yield env.timeout(transfer_duration)

            # Normalise transfer probabilities
            transfer_probs = np.array([P_TRANSFER_TO_MOVEMENT, P_TRANSFER_TO_ADOPTION, P_TRANSFER_TO_TRANSFER])
            transfer_probs = transfer_probs / transfer_probs.sum()
            current_stage = np.random.choice(['Movement', 'Adoption', 'Transfer'], p=transfer_probs)

        elif current_stage == 'Adoption':
            # Allow return only if under max_returns threshold
            if return_count < max_returns and np.random.random() < P_ADOPTION_TO_RETURN:
                return_duration = np.random.exponential(1 / RETURN_RATE)
                yield env.timeout(return_duration)
                return_count += 1

                # Normalise return probabilities
                return_probs = np.array([P_RETURN_TO_ADOPTION, P_RETURN_TO_MOVEMENT])
                return_probs = return_probs / return_probs.sum()
                current_stage = np.random.choice(['Adoption', 'Movement'], p=return_probs)
            else:
                break

    total_los = env.now - arrival_time
    results.append({'dog_id': dog_id, 'los': total_los, 'movements': movement_count})


# Run Monte Carlo with multiple return cycles
N_REPLICATIONS = 100
multi_return_results = []

for rep in range(N_REPLICATIONS):
    np.random.seed(rep)
    env = simpy.Environment()
    rep_results = []

    def arrival_process_multi(env, results):
        dog_id = 0
        while True:
            yield env.timeout(np.random.exponential(1 / ARRIVAL_RATE))
            env.process(dog_journey_multi_return(env, dog_id, results))
            dog_id += 1

    env.process(arrival_process_multi(env, rep_results))
    env.run(until=SIM_DURATION)
    multi_return_results.extend(rep_results)

multi_df = pd.DataFrame(multi_return_results)

# Compare LOS band distributions
bins = [0, 60, 182, float('inf')]
labels = ['0-60 days (Normal)', '60-182 days (Medium)', '183+ days (Long-term)']
multi_df['LOS Band'] = pd.cut(multi_df['los'], bins=bins, labels=labels)

sim_band_props_multi = multi_df['LOS Band'].value_counts(normalize=True).sort_index() * 100

comparison_multi = pd.DataFrame({
    'Real Data (%)': real_band_props,
    'Single Return (%)': sim_band_props_v3.round(1),
    'Multiple Returns (%)': sim_band_props_multi.round(1)
})

print("Diagnostic — Multiple Return Cycles vs Single Return vs Real Data:")
print(comparison_multi)
print(f"\nMultiple Return Model — Mean LOS: {multi_df['los'].mean():.2f} days")